## 04_city_breakdown — 都市別 Coles/Woolworths 比較

**入力**
- `output/data/coles_locations.csv`      — Coles 店舗座標 (N_C = 685)
- `output/data/woolworths_locations.csv` — Woolworths 店舗座標 (N_W = 1,039)

**出力**
- `output/analysis/city_breakdown.csv`  — city, n_coles, n_woolworths, ratio_cw

**前提**
- `00_random_catalog.ipynb` 不要（立地データのみ使用）

**検証仮説**

| 仮説 | 内容 | 期待 |
|---|---|---|
| A-1 | Retail BAO 非対称：Melbourne で Coles 優位 | C/W > 1 |
| A-2 | Retail BAO 非対称：Sydney で Woolworths 優位 | C/W < 1 |
| B   | 114 km バンプ：Newcastle で Coles 特異集中 | C/W > 1 |
| C   | 239 km バンプ：Canberra で Woolworths 優位 | C/W < 1 |

In [1]:
%use dataframe
%use lets-plot

In [2]:
// --- データ読み込み ---
val dfC = DataFrame.readCSV("./output/data/coles_locations.csv")
val dfW = DataFrame.readCSV("./output/data/woolworths_locations.csv")

val nC = dfC.rowsCount();  val nW = dfW.rowsCount()
val nationalRatio = nC.toDouble() / nW
println("全国: Coles $nC 店舗 / Woolworths $nW 店舗  (C/W = ${"%.3f".format(nationalRatio)})")

全国: Coles 685 店舗 / Woolworths 1039 店舗  (C/W = 0.659)


In [3]:
// --- 都市バウンディングボックス定義 ---
data class CityBox(val name: String, val latMin: Double, val latMax: Double,
                   val lonMin: Double, val lonMax: Double)

val cities = listOf(
    CityBox("Sydney",      -34.2, -33.4, 150.5, 151.4),
    CityBox("Melbourne",   -38.2, -37.4, 144.5, 145.8),
    CityBox("Brisbane",    -27.8, -27.2, 152.7, 153.3),
    CityBox("Adelaide",    -35.2, -34.6, 138.4, 138.9),
    CityBox("Perth",       -32.2, -31.7, 115.7, 116.1),
    CityBox("Canberra",    -35.5, -35.2, 148.9, 149.3),
    CityBox("Newcastle",   -33.1, -32.7, 151.5, 151.9),
    CityBox("Gold Coast",  -28.2, -27.9, 153.3, 153.6),
    CityBox("Hobart",      -43.0, -42.7, 147.0, 147.5),
    CityBox("Darwin",      -12.6, -12.3, 130.8, 131.1)
)

fun countInBox(df: org.jetbrains.kotlinx.dataframe.DataFrame<*>, box: CityBox): Int =
    df.rows().count { row ->
        val lat = row["lat"] as Double; val lon = row["lon"] as Double
        lat in box.latMin..box.latMax && lon in box.lonMin..box.lonMax
    }

data class CityResult(val name: String, val nColes: Int, val nWool: Int, val ratio: Double)

val results: List<CityResult> = cities.map { box ->
    val nc = countInBox(dfC, box);  val nw = countInBox(dfW, box)
    CityResult(box.name, nc, nw, if (nw > 0) nc.toDouble() / nw else Double.NaN)
}

println("%-12s  %6s  %12s  %8s  %s".format("都市", "Coles", "Woolworths", "C/W", "優位"))
println("-".repeat(52))
results.forEach { r ->
    val winner = when {
        r.ratio.isNaN()   -> "—"
        r.ratio > nationalRatio * 1.05 -> "Coles ↑"
        r.ratio < nationalRatio * 0.95 -> "Woolworths ↑"
        else                           -> "拮抗"
    }
    println("%-12s  %6d  %12d  %8.3f  %s".format(r.name, r.nColes, r.nWool, r.ratio, winner))
}

都市             Coles    Woolworths       C/W  優位
----------------------------------------------------
Sydney           114           160     0.713  Coles ↑
Melbourne        137           177     0.774  Coles ↑
Brisbane          56           106     0.528  Woolworths ↑
Adelaide          34            47     0.723  Coles ↑
Perth             53            65     0.815  Coles ↑
Canberra          11            16     0.688  拮抗
Newcastle         16            21     0.762  Coles ↑
Gold Coast        20            22     0.909  Coles ↑
Hobart             5            15     0.333  Woolworths ↑
Darwin             6             8     0.750  Coles ↑


In [4]:
// --- 仮説 A/B/C 検証 ---
val cityMap = results.associateBy { it.name }

fun testHypothesis(label: String, city: String, expectedHigher: String) {
    val r = cityMap[city] ?: return
    val actual = when {
        r.ratio > nationalRatio * 1.05 -> "Coles"
        r.ratio < nationalRatio * 0.95 -> "Woolworths"
        else                           -> "拮抗"
    }
    val mark = if (actual == expectedHigher) "✓ 支持" else "✗ 否定"
    println("[$mark]  $label")
    println("         $city: C=${r.nColes}, W=${r.nWool}, C/W=${"%.3f".format(r.ratio)} (全国比 ${"%.3f".format(nationalRatio)}) → $actual 優位")
    println()
}

testHypothesis("仮説 A-1: BAO → Melbourne で Coles 優位",     "Melbourne",  "Coles")
testHypothesis("仮説 A-2: BAO → Sydney で Woolworths 優位",   "Sydney",     "Woolworths")
testHypothesis("仮説 B:   114 km バンプ → Newcastle で Coles", "Newcastle",  "Coles")
testHypothesis("仮説 C:   239 km バンプ → Canberra で Woolworths", "Canberra", "Woolworths")

[✓ 支持]  仮説 A-1: BAO → Melbourne で Coles 優位
         Melbourne: C=137, W=177, C/W=0.774 (全国比 0.659) → Coles 優位

[✗ 否定]  仮説 A-2: BAO → Sydney で Woolworths 優位
         Sydney: C=114, W=160, C/W=0.713 (全国比 0.659) → Coles 優位

[✓ 支持]  仮説 B:   114 km バンプ → Newcastle で Coles
         Newcastle: C=16, W=21, C/W=0.762 (全国比 0.659) → Coles 優位

[✗ 否定]  仮説 C:   239 km バンプ → Canberra で Woolworths
         Canberra: C=11, W=16, C/W=0.688 (全国比 0.659) → 拮抗 優位



In [5]:
// --- バープロット: C/W 比 ---
val validCities = results.filter { !it.ratio.isNaN() && (it.nColes + it.nWool) >= 3 }

letsPlot(mapOf("city" to validCities.map { it.name }, "ratio" to validCities.map { it.ratio })) +
    geomBar(stat = Stat.identity, width = 0.6) { x = "city"; y = "ratio"; fill = "ratio" } +
    geomHLine(yintercept = nationalRatio, linetype = "dashed", color = "#CC4444", size = 1.0) +
    geomHLine(yintercept = 1.0, linetype = "dotted", color = "#666666") +
    scaleXDiscrete(name = "都市") +
    scaleYContinuous(name = "Coles / Woolworths 比") +
    scaleFillGradient2(low = "#4682B4", mid = "#EEEEEE", high = "#E87040", midpoint = nationalRatio) +
    ggtitle("都市別 Coles/Woolworths 比",
            "赤破線: 全国比 (${"%.3f".format(nationalRatio)}) | 点線: 等数 (1.0)") +
    ggsize(800, 380)

Sydney 
 
 
 
 
 
 
 
 
 Melbourne 
 
 
 
 
 
 
 
 
 Brisbane 
 
 
 
 
 
 
 
 
 Adelaide 
 
 
 
 
 
 
 
 
 Perth 
 
 
 
 
 
 
 
 
 Canberra 
 
 
 
 
 
 
 
 
 Newcastle 
 
 
 
 
 
 
 
 
 Gold Coast 
 
 
 
 
 
 
 
 
 Hobart 
 
 
 
 
 
 
 
 
 Darwin 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 1 
 
 
 
 
 
 
 
 
 都市別 Coles/Woolworths 比 
 
 
 
 
 赤破線: 全国比 (0.659) | 点線: 等数 (1.0) 
 
 
 
 
 Coles / Woolworths 比 
 
 
 
 
 都市 
 
 
 
 
 
 
 
 
 ratio 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 
 
 
 
 0.5 
 
 
 
 
 
 
 
 
 0.6 
 
 
 
 
 
 
 
 
 0.7 
 
 
 
 
 
 
 
 
 0.8 
 
 
 
 
 
 
 
 
 0.9

In [6]:
// --- CSV 出力 ---
java.io.File("./output/analysis/city_breakdown.csv").bufferedWriter().use { w ->
    w.appendLine("city,n_coles,n_woolworths,ratio_cw")
    results.forEach { r -> w.appendLine("${r.name},${r.nColes},${r.nWool},${r.ratio}") }
}
println("保存: output/analysis/city_breakdown.csv")

保存: output/analysis/city_breakdown.csv
